# UMUD Challenge - Submission 2: Segmentation-First Pipeline
This notebook trains a U-Net to predict Aponeurosis and Fascicle masks from the ultrasound images, and then mathematically derives PA, FL, and MT directly from those predicted masks using geometric algorithms.

**Instructions for Kaggle:**
1. Turn on the **GPU** (P100 or T4x2).
2. Ensure the dataset is added to `/kaggle/input/`.
3. Run all cells. The final submission will be saved to `/kaggle/working/submission.csv`.

In [30]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from pathlib import Path
import glob
from tqdm.auto import tqdm
import math

# Configuration
# ==========================================
# Dynamically locate the directory containing 'apo_imgs_v1'
matching_dirs = glob.glob('/kaggle/input/**/apo_imgs_v1', recursive=True)

if matching_dirs:
    APO_IMG_DIR = Path(matching_dirs[0])
    COMPETITION_DIR = APO_IMG_DIR.parent
else:
    # Fallback to default Kaggle input structure
    COMPETITION_DIR = Path(glob.glob('/kaggle/input/*')[0]) if glob.glob('/kaggle/input/*') else Path('.')
    APO_IMG_DIR = COMPETITION_DIR / 'apo_imgs_v1'

APO_MASK_DIR  = COMPETITION_DIR / 'apo_masks_v1'
FASC_MASK_DIR = COMPETITION_DIR / 'fasc_masks_v1'
TEST_DIR      = COMPETITION_DIR / 'test_images_v2'
OUT_DIR       = Path('/kaggle/working')

print("Checking directories...")
print(f"Dataset root: {COMPETITION_DIR} (Exists: {COMPETITION_DIR.exists()})")
print(f"Apo Imgs: {APO_IMG_DIR} (Exists: {APO_IMG_DIR.exists()})")
print(f"Apo Masks: {APO_MASK_DIR} (Exists: {APO_MASK_DIR.exists()})")
print(f"Fasc Masks: {FASC_MASK_DIR} (Exists: {FASC_MASK_DIR.exists()})")

if not APO_IMG_DIR.exists():
    raise FileNotFoundError(f"Could not find the dataset at {APO_IMG_DIR}. Please check dataset path in /kaggle/input/")

IMG_SIZE = 256  # Train at 256x256 for speed/memory, then upscale masks
BATCH_SIZE = 16
EPOCHS = 15     # Adjust based on time limit. 15 is a solid baseline for U-Net
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# All test images and our characterized known scales share exactly this resolution:
SCALE_MM_PER_PIXEL = 0.06979

KeyboardInterrupt: 

In [ ]:
import os

class UltrasoundDataset(Dataset):
    def __init__(self, record_ids, augment=False):
        self.record_ids = record_ids
        self.augment = augment

    def __len__(self):
        return len(self.record_ids)

    def __getitem__(self, idx):
        rec = self.record_ids[idx]
        
        # Helper to search recursively inside nested subfolders
        def find_file(directory, filename_stem):
            for p in directory.rglob('*'):
                if p.is_file() and filename_stem in p.stem and p.suffix.lower() in ['.tif', '.png', '.jpg', '.jpeg']:
                    return str(p)
            return None

        img_path = find_file(APO_IMG_DIR, rec)
        apo_mask_path = find_file(APO_MASK_DIR, rec)
        fasc_mask_path = find_file(FASC_MASK_DIR, rec)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) if img_path else None
        apo = cv2.imread(apo_mask_path, cv2.IMREAD_GRAYSCALE) if apo_mask_path else None
        fasc = cv2.imread(fasc_mask_path, cv2.IMREAD_GRAYSCALE) if fasc_mask_path else None

        # Fallback empty arrays if image loading fails
        if img is None: 
            img = np.zeros((800, 1200), dtype=np.uint8)
        
        h, w = img.shape[:2]
        if apo is None: apo = np.zeros((h, w), dtype=np.uint8)
        if fasc is None: fasc = np.zeros((h, w), dtype=np.uint8)

        # Resize for UNet
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        apo = cv2.resize(apo, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        fasc = cv2.resize(fasc, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # Normalize
        img = img.astype(np.float32) / 255.0
        apo = (apo > 127).astype(np.float32)
        fasc = (fasc > 127).astype(np.float32)

        img = torch.from_numpy(img).unsqueeze(0)
        mask = torch.stack([torch.from_numpy(apo), torch.from_numpy(fasc)], dim=0)

        if self.augment and torch.rand(1).item() > 0.5:
            img = TF.hflip(img)
            mask = TF.hflip(mask)

        return img, mask


def get_clean_stems_recursive(directory):
    stems = set()
    # rglob checks inside nested directories
    for p in directory.rglob('*'):
        if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg', '.jpeg']:
            clean_name = p.stem.replace('_apo', '').replace('_fasc', '').replace('_mask', '')
            stems.add(clean_name)
    return stems

apo_imgs = get_clean_stems_recursive(APO_IMG_DIR)
apo_masks = get_clean_stems_recursive(APO_MASK_DIR)
fasc_masks = get_clean_stems_recursive(FASC_MASK_DIR)

# Find records present across images and mask directories
valid_records = sorted(list(apo_imgs & (apo_masks | fasc_masks)))
print(f"Total overlapping training records: {len(valid_records)}")

if len(valid_records) == 0:
    raise ValueError("Zero valid records found! Check dataset contents.")

val_size = max(1, int(len(valid_records) * 0.15))
train_records = valid_records[:-val_size]
val_records = valid_records[-val_size:]

train_ds = UltrasoundDataset(train_records, augment=True)
val_ds = UltrasoundDataset(val_records, augment=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=2):
        super().__init__()
        features = [32, 64, 128, 256]
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        # Final
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return self.final_conv(x)


In [ ]:
model = UNet(in_channels=1, out_channels=2).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda')

print("Starting U-Net Training...")
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, masks)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * imgs.size(0)
    
    train_loss /= len(train_ds)
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_dl, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = criterion(preds, masks)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_ds)
    
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), OUT_DIR / 'best_unet.pth')

print("Training Complete!")


In [ ]:
def measure_geometry(apo_mask, fasc_mask, scale_mm_per_pixel=0.06979):
    # apo_mask, fasc_mask are binary numpy arrays (0 or 255) of size 800x1200 (original test dims)
    
    ### 1. PENNATION ANGLE (PA) ###
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fasc_mask, 8)
    angles, weights = [], []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 50: continue
        pts = np.column_stack(np.where(labels == i)).astype(np.float64)
        if len(pts) < 5: continue
        cov = np.cov(pts.T)
        if cov.shape != (2, 2): continue
        evals, evecs = np.linalg.eigh(cov)
        principal = evecs[:, np.argmax(evals)] # [y, x]
        angle = np.degrees(np.arctan2(abs(principal[0]), abs(principal[1])))
        angles.append(angle)
        weights.append(float(area))
        
    pa_deg = np.average(angles, weights=weights) if angles else 15.0 # fallback
    
    ### 2. MUSCLE THICKNESS (MT) ###
    # Identify superficial (top) and deep (bottom) aponeuroses
    n_apo, apo_labels, apo_stats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    if n_apo >= 3:
        # Sort components by area, ignore background (0)
        comps = sorted([(i, apo_stats[i, cv2.CC_STAT_AREA]) for i in range(1, n_apo)], key=lambda x: x[1], reverse=True)
        c1, c2 = comps[0][0], comps[1][0]
        y1 = np.mean(np.where(apo_labels == c1)[0])
        y2 = np.mean(np.where(apo_labels == c2)[0])
        
        sup_c = c1 if y1 < y2 else c2
        deep_c = c2 if y1 < y2 else c1
        
        sup_mask = (apo_labels == sup_c).astype(np.uint8)
        deep_mask = (apo_labels == deep_c).astype(np.uint8)
        
        # Inner boundaries (bottom of superficial, top of deep)
        sup_y = np.argmax(sup_mask[::-1], axis=0)
        sup_bottom = sup_mask.shape[0] - 1 - sup_y
        sup_bottom = np.where(sup_mask.any(axis=0), sup_bottom, np.nan)
        
        deep_top = np.argmax(deep_mask, axis=0)
        deep_top = np.where(deep_mask.any(axis=0), deep_top, np.nan)
        
        # Valid overlap
        thickness_profile = deep_top - sup_bottom
        valid = ~np.isnan(thickness_profile) & (thickness_profile > 0)
        
        if np.sum(valid) > 50:
            mt_px = np.nanmedian(thickness_profile[valid])
            mt_mm = mt_px * scale_mm_per_pixel
        else:
            mt_mm = 20.0 # fallback
    else:
        mt_mm = 20.0 # fallback
        
    ### 3. FASCICLE LENGTH (FL) ###
    fl_mm = mt_mm / math.sin(math.radians(pa_deg))
    
    # Clip to anatomically reasonable ranges
    pa_deg = np.clip(pa_deg, 3.0, 45.0)
    fl_mm = np.clip(fl_mm, 20.0, 200.0)
    mt_mm = np.clip(mt_mm, 5.0, 40.0)
    
    return pa_deg, fl_mm, mt_mm


In [31]:
# Final Inference & Submission Generator

results = []

# Collect all test image paths once upfront
test_files = sorted([p for p in TEST_DIR.rglob('*') if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg', '.jpeg']])

print(f"Found {len(test_files)} test images for inference.")

model.eval()
with torch.no_grad():
    for test_path in tqdm(test_files, desc="Processing Test Images"):
        # Load image in grayscale
        img_orig = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
        if img_orig is None:
            continue
            
        orig_h, orig_w = img_orig.shape[:2]
        
        # Preprocess for model input
        img_resized = cv2.resize(img_orig, (IMG_SIZE, IMG_SIZE))
        img_tensor = img_resized.astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0).to(DEVICE)
        
        # Predict mask
        with torch.amp.autocast('cuda'):
            preds = torch.sigmoid(model(img_tensor))
        
        preds_np = (preds.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
        apo_pred = preds_np[0]
        fasc_pred = preds_np[1]
        
        # Resize masks back to original image size (width, height)
        apo_full = cv2.resize(apo_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        fasc_full = cv2.resize(fasc_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        
        # Calculate geometric measurements
        pa, fl, mt = measure_geometry(apo_full, fasc_full)
        
        # Ensure image_id retains extension (e.g. IMG_00001.tif)
        image_id = test_path.name if test_path.suffix else f"{test_path.stem}.tif"
        
        results.append({
            'image_id': image_id,
            'pa_deg': round(float(pa), 3),
            'fl_mm': round(float(fl), 3),
            'mt_mm': round(float(mt), 3)
        })

# Create final submission DataFrame
sub_df = pd.DataFrame(results)

# Save with semicolon separator as expected by the competition evaluator
# Save with standard comma separator! Kaggle's backend expects commas, NOT semicolons!
sub_df.to_csv(OUT_DIR / 'submission.csv', sep=',', index=False)

print(f"\nSuccessfully generated submission.csv with {len(sub_df)} rows!")
print(sub_df.head())

Found 309 test images for inference.


Processing Test Images:   0%|          | 0/309 [00:00<?, ?it/s]


Successfully generated submission.csv with 309 rows!
        image_id  pa_deg   fl_mm   mt_mm
0  IMG_00001.tif    15.0  92.220  23.868
1  IMG_00002.tif    15.0  73.614  19.053
2  IMG_00003.tif    15.0  84.669  21.914
3  IMG_00004.tif    15.0  96.264  24.915
4  IMG_00005.tif    15.0  77.659  20.100
